# §39 step 1b — **ACCIDENT vs TAD: contact sheet of the highest-score pairs** (CPU)

Why: the first dedup STOPPED (`REPORTS/accident_dedup.md`: tau 0.7911 flags 52.2 % of the clips; tau sits below the cross-source noise floor, `KIM_RUN.md` §39 D). The old sheet drew the first 60 flagged pairs **in alphabetical order**, not by score. User's choice (2026-11-22, option 2): look at **every clip whose TAD top-1 is ≥ 0.87** (88 clips / 86 sources), **highest score first**, and pick the threshold by eye, at the score where pairs stop being one camera.

Reads `Thesis-Final/outputs/kim_accident/REPORTS/accident_dedup.json` (no re-dedup, no GPU), the ACCIDENT mp4 on Drive, and downloads only the TAD images it draws from Kaggle. Writes `REPORTS/accident_dedup_high.md`, `accident_dedup_high_pairs.json`, `accident_dedup_high_p01.png ...` (8 pairs per page). Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first (needs `accident_dedup.high_score_pairs`).

**What to tell Claude:** the rank numbers (printed in each title and in the .md table) of the pairs that are **the same physical camera** (same mount, same road geometry, same fixed objects), not just the same kind of scene. Rank 1 is the highest score; the cut-off is where they stop.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

DRIVE_F = '/content/drive/MyDrive/Thesis-Final'
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = '/content/drive/MyDrive/Thesis/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = '/content/drive/MyDrive/Thesis/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

REPO = Path(DRIVE_F) / 'kat-vad'
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert 'def high_score_pairs' in (REPO / 'core/tools/accident_dedup.py').read_text(), \
    'accident_dedup.py has no high_score_pairs -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
ACC     = Path(DRIVE_F) / 'data' / constants.ACCIDENT_DATASET
VIDEOS  = ACC / constants.ACCIDENT_VIDEOS_DIRNAME
REPORTS = constants.OUTPUT_ROOT / constants.KIM_ACCIDENT_OUTPUT_DIRNAME / 'REPORTS'
DEDUP   = REPORTS / f'{constants.ACCIDENT_DEDUP_STEM}.json'
LOCAL   = Path('/content/accident_high')
LOCAL.mkdir(parents=True, exist_ok=True)
for name, p in (('dedup json', DEDUP), ('ACCIDENT mp4 dir', VIDEOS)):
    assert p.exists(), f'missing {name}: {p} -- paste to Claude'
print(f'code {REPO} | dedup {DEDUP} | videos {VIDEOS}')

In [ ]:
%%bash
pip install -q kagglehub av
python -c "import kagglehub, av; print('deps OK')"

In [ ]:
import av  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import kagglehub  # noqa: E402

from core.tools import accident_dedup  # noqa: E402
from core.tools.accident_build import source_of  # noqa: E402
from core.tools.accident_extract import cfr_index  # noqa: E402

result = json.loads(DEDUP.read_text())
pairs = accident_dedup.high_score_pairs(result, constants.ACCIDENT_DEDUP_SHEET_MIN_SCORE,
                                        constants.ACCIDENT_DEDUP_ROW_STEP, constants.FRAME_STRIDE)
stem = constants.ACCIDENT_DEDUP_SHEET_STEM
print(f"{len(pairs)} clips with TAD top-1 >= {constants.ACCIDENT_DEDUP_SHEET_MIN_SCORE} "
      f"({len({source_of(p['acc_id']) for p in pairs})} sources); tau was {result['tau']:.4f}")

lines = ['# ACCIDENT vs TAD: pairs with TAD top-1 >= ' f"{constants.ACCIDENT_DEDUP_SHEET_MIN_SCORE}", '',
         '| rank | ACCIDENT clip | source | TAD video | score |', '|---|---|---|---|---|']
lines += [f"| {i} | {p['acc_id']} | {source_of(p['acc_id'])} | {p['tad_id']} | {p['score']:.4f} |"
          for i, p in enumerate(pairs, 1)]
(LOCAL / f'{stem}.md').write_text('\n'.join(lines) + '\n')
(LOCAL / f'{stem}_pairs.json').write_text(json.dumps(pairs, indent=1))


def acc_frame(clip_id, k):
    with av.open(str(VIDEOS / f'{clip_id}.mp4')) as c:
        frames = [(f.time, f.to_ndarray(format='rgb24')) for f in c.decode(c.streams.video[0])]
    idx = cfr_index(np.array([t for t, _ in frames]))
    return frames[int(idx[min(k, len(idx) - 1)])][1]


def tad_frame(tad_id, k):
    # one TAD image, downloaded alone from Kaggle (TAD frames are not on Drive; only the sheet needs pixels)
    kind = 'normal' if tad_id.startswith('Normal') else 'abnormal'
    for j in (k, max(0, k - 8)):              # k = s8 row x 8; one step back if the folder ends early
        try:
            return plt.imread(kagglehub.dataset_download(constants.TAD_KAGGLE_HANDLE,
                                                         path=f'TAD/frames/{kind}/{tad_id}.mp4/{j}.jpg'))
        except Exception as exc:  # noqa: BLE001 -- a missing image must not stop the sheet
            print(f'TAD {tad_id} frame {j}: {exc}')
    return np.zeros((8, 8, 3))


per = constants.ACCIDENT_DEDUP_SHEET_ROWS_PER_PAGE
pages = []
for page, start in enumerate(range(0, len(pairs), per), 1):
    chunk = pairs[start:start + per]
    fig, axes = plt.subplots(len(chunk), 2, figsize=(9, 2.7 * len(chunk)), squeeze=False)
    for ax, (i, p) in zip(axes, enumerate(chunk, start + 1)):
        ax[0].imshow(acc_frame(p['acc_id'], p['acc_frame']))
        ax[0].set_title(f"#{i}  {p['acc_id']} f{p['acc_frame']}  score {p['score']:.3f}", fontsize=9)
        ax[1].imshow(tad_frame(p['tad_id'], p['tad_frame']))
        ax[1].set_title(f"TAD {p['tad_id']} f{p['tad_frame']}", fontsize=9)
        for a in ax:
            a.axis('off')
    fig.tight_layout()
    png = LOCAL / f'{stem}_p{page:02d}.png'
    fig.savefig(png, dpi=100)
    pages.append(png)
    plt.show()
    plt.close(fig)
print(f'{len(pages)} pages')

In [ ]:
import time  # noqa: E402


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


for f in sorted(LOCAL.iterdir()):
    publish(f, REPORTS / 'high' / f.name)
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f"published {len(list(LOCAL.iterdir()))} files to {REPORTS / 'high'}")
print('Download REPORTS/high/ (pngs + .md + pairs.json) and tell Claude the ranks that are the SAME camera')